In [54]:
# Environment setup
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import numpy as np

print('='*60)
print('Environment')
print('='*60)
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('='*60)

Environment
PyTorch: 2.9.1+rocm6.4
GPU: AMD Instinct MI250X/MI250


In [55]:
# Imports
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from transformers import (
    RobertaTokenizerFast, RobertaForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print('✓ Imports complete')

✓ Imports complete


In [56]:
# Load data
BASE = Path('../..')
TRAIN_FILE = BASE / 'train_rehydrated.jsonl'

def load_data(file_path):
    data = []
    with open(file_path) as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

train_data = load_data(TRAIN_FILE)
print(f'✓ Loaded {len(train_data)} examples')
print(f'Sample: {len(train_data[0].get("markers", []))} markers')

✓ Loaded 4316 examples
Sample: 5 markers


In [57]:
# Label mapping - BIO tagging
MARKER_TYPES = ['Action', 'Actor', 'Effect', 'Evidence', 'Victim']

label_list = ['O']
for mt in MARKER_TYPES:
    label_list.extend([f'B-{mt}', f'I-{mt}'])

label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}
num_labels = len(label_list)

print(f'Labels: {num_labels} ({len(MARKER_TYPES)} marker types with BIO tagging)')

Labels: 11 (5 marker types with BIO tagging)


In [58]:
# Tokenizer and alignment
MODEL_NAME = 'roberta-large'
MAX_LENGTH = 512

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tok = tokenizer(
        examples['text'], truncation=True, max_length=MAX_LENGTH,
        return_offsets_mapping=True, is_split_into_words=False
    )
    labels, all_markers = [], examples.get('markers', [])
    for i, offsets in enumerate(tok['offset_mapping']):
        ex_labels = [label_to_id['O']] * len(offsets)
        # Handle None markers (for test data without labels)
        marker_list = all_markers[i] if (i < len(all_markers) and all_markers[i] is not None) else []
        markers = sorted(marker_list, key=lambda x: x['startIndex']) if marker_list else []
        for m in markers:
            b_lbl, i_lbl = label_to_id.get(f"B-{m['type']}"), label_to_id.get(f"I-{m['type']}")
            if b_lbl is None: continue
            first = True
            for ti, (s, e) in enumerate(offsets):
                if s is None: continue
                if s < m['endIndex'] and e > m['startIndex']:
                    ex_labels[ti] = b_lbl if first else (i_lbl if ex_labels[ti] == label_to_id['O'] else ex_labels[ti])
                    first = False
        labels.append(ex_labels)
    tok['labels'] = labels
    return tok

print(f'✓ Tokenizer ready')

✓ Tokenizer ready


In [59]:
# Train/val split
VAL_SPLIT = 0.1
train_idx, val_idx = train_test_split(range(len(train_data)), test_size=VAL_SPLIT, random_state=42)
train_ds = Dataset.from_list([train_data[i] for i in train_idx])
val_ds = Dataset.from_list([train_data[i] for i in val_idx])

print(f'Tokenizing...')
train_ds = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['_id', 'text', 'markers', 'subreddit', 'conspiracy', 'annotator'])
val_ds = val_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['_id', 'text', 'markers', 'subreddit', 'conspiracy', 'annotator'])

print(f'✓ Train: {len(train_ds)}, Val: {len(val_ds)}')

Tokenizing...


Map:  26%|██▌       | 1000/3884 [00:00<00:00, 3435.56 examples/s]

Map: 100%|██████████| 432/432 [00:00<00:00, 3412.94 examples/s]

✓ Train: 3884, Val: 432


In [60]:
# Model configuration
from transformers import RobertaConfig

config = RobertaConfig.from_pretrained(MODEL_NAME)
config.num_labels = num_labels
config.id2label = id_to_label
config.label2id = label_to_id

model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, config=config)

if torch.cuda.is_available():
    model = model.to('cuda')
    print('✓ Model on GPU')

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
TARGET_MODULES = ['query', 'value', 'key', 'dense']

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT, target_modules=TARGET_MODULES, bias='none'
)
model = get_peft_model(model, lora_config)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ LoRA applied: {train_p:,}/{total_p:,} params ({100*train_p/total_p:.2f}%)')

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model on GPU
✓ LoRA applied: 7,089,163/361,410,582 params (1.96%)


In [61]:
# Metrics
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    true_l, pred_l = [], []
    for p, l in zip(preds, labels):
        for pi, li in zip(p, l):
            if li != -100:
                true_l.append(id_to_label[li])
                pred_l.append(id_to_label[pi])
    f1_micro = f1_score(true_l, pred_l, average='micro')
    ent_l = [l for l in true_l if l != 'O']
    ent_p = [p for p, l in zip(pred_l, true_l) if l != 'O']
    f1_ent = f1_score(ent_l, ent_p, average='micro') if ent_l else 0.0
    return {'f1_overall': f1_micro, 'f1_entity': f1_ent}

print('✓ Metrics defined')

✓ Metrics defined


In [71]:
# Training
BATCH_SIZE, GRAD_ACC, LR, EPOCHS = 15, 2, 5e-4, 10
WEIGHT_DECAY, WARMUP = 0,0 #0.01, 0.1

OUTPUT_DIR = BASE / 'Markers-Extraction' / 'models' / 'roberta-large-markers-lora'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

use_bf16 = False
if torch.cuda.is_available():
    try:
        _ = torch.tensor([1.0], dtype=torch.bfloat16)
        use_bf16 = True
    except:
        pass

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1_entity', greater_is_better=True,
    logging_steps=50, report_to='none', seed=42,
    gradient_accumulation_steps=GRAD_ACC, bf16=use_bf16,
    gradient_checkpointing=True, save_total_limit=2
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)
early_stop = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics, callbacks=[early_stop]
)

print('='*60)
print('Starting Training')
print('='*60)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
eval_result = trainer.evaluate()

print('\n' + '='*60)
print('Training Complete!')
print('='*60)
print(f'Loss: {train_result.training_loss:.4f}')
for k, v in eval_result.items():
    if k.startswith('eval_'):
        print(f'{k}: {v:.4f}')
print('='*60)

Starting Training


Epoch,Training Loss,Validation Loss,F1 Overall,F1 Entity
1,No log,0.922493,0.733540,0.090883
2,No log,0.935430,0.735659,0.092017
3,0.869500,0.934650,0.737324,0.112801
4,0.869500,0.962696,0.723399,0.186585
5,0.869500,0.936080,0.729428,0.187246
6,0.667100,0.949570,0.730412,0.169391
7,0.667100,0.975047,0.714369,0.226169
8,0.667100,0.993214,0.707078,0.241663
9,0.584400,1.004810,0.711367,0.236089
10,0.584400,1.002330,0.723980,0.218989



Training Complete!
Loss: 0.6887
eval_loss: 0.9932
eval_f1_overall: 0.7071
eval_f1_entity: 0.2417
eval_runtime: 2.8541
eval_samples_per_second: 151.3640
eval_steps_per_second: 1.4020


In [72]:
# Evaluation
print('Generating predictions...')
predictions = trainer.predict(val_ds)
pred_labels = np.argmax(predictions.predictions, axis=2)
true_labels = predictions.label_ids

true_flat, pred_flat = [], []
for p, l in zip(pred_labels, true_labels):
    for pi, li in zip(p, l):
        if li != -100:
            true_flat.append(id_to_label[li])
            pred_flat.append(id_to_label[pi])

print('\n' + '='*60)
print('Classification Report (Validation)')
print('='*60)
print(classification_report(true_flat, pred_flat))
print('='*60)

Generating predictions...



Classification Report (Validation)
              precision    recall  f1-score   support

    B-Action       0.34      0.17      0.23       490
     B-Actor       0.44      0.38      0.41       571
    B-Effect       0.16      0.04      0.07       378
  B-Evidence       0.31      0.07      0.12       380
    B-Victim       0.29      0.23      0.25       302
    I-Action       0.28      0.29      0.28      2140
     I-Actor       0.45      0.37      0.41      1270
    I-Effect       0.29      0.26      0.28      2089
  I-Evidence       0.32      0.16      0.21      2558
    I-Victim       0.34      0.23      0.27       407
           O       0.80      0.88      0.84     29057

    accuracy                           0.71     39642
   macro avg       0.37      0.28      0.31     39642
weighted avg       0.67      0.71      0.69     39642



In [73]:
# Save model
final_dir = OUTPUT_DIR / 'final_model'
final_dir.mkdir(exist_ok=True)

trainer.model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

label_mapping = {'label_to_id': label_to_id, 'id_to_label': id_to_label, 'marker_types': MARKER_TYPES}
with open(final_dir / 'label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f'✓ Model saved to: {final_dir}')
print('✓ Training complete!')

✓ Model saved to: ../../Markers-Extraction/models/roberta-large-markers-lora/final_model
✓ Training complete!


## Inference on Dev/Test Data

Generate predictions for submission to CodaBench.

In [74]:
# Load test/dev data for inference
TEST_FILE = BASE / 'dev_rehydrated.jsonl'

print('Loading test data...')
test_data = load_data(TEST_FILE)
print(f'✓ Loaded {len(test_data)} test examples')

# Keep original data for creating submission
test_data_original = test_data.copy()

Loading test data...
✓ Loaded 100 test examples


In [75]:
# Tokenize test data
print('Tokenizing test data...')
test_ds = Dataset.from_list(test_data)

# Only remove columns that exist in the test dataset
cols_to_remove = [col for col in ['_id', 'text', 'markers', 'subreddit', 'conspiracy', 'annotator'] 
                  if col in test_ds.column_names]

test_ds_tokenized = test_ds.map(
    tokenize_and_align_labels, 
    batched=True,
    remove_columns=cols_to_remove
)

print(f'✓ Test dataset ready: {len(test_ds_tokenized)} examples')

Tokenizing test data...


Map: 100%|██████████| 100/100 [00:00<00:00, 3393.15 examples/s]

✓ Test dataset ready: 100 examples


In [76]:
# Generate predictions
print('Generating predictions...')
test_predictions = trainer.predict(test_ds_tokenized)
test_pred_labels = np.argmax(test_predictions.predictions, axis=2)

print(f'✓ Generated predictions for {len(test_pred_labels)} examples')

Generating predictions...


✓ Generated predictions for 100 examples


In [77]:
# Convert token predictions to character spans
def extract_markers_from_predictions(pred_labels, offset_mappings, texts, id_to_label):
    """Convert token-level BIO predictions to character-level marker spans"""
    all_predictions = []
    
    for example_idx, (pred, offsets, text) in enumerate(zip(pred_labels, offset_mappings, texts)):
        markers = []
        current_marker = None
        
        for token_idx, (label_id, offset) in enumerate(zip(pred, offsets)):
            if offset[0] is None or offset[1] is None:
                continue
            
            label = id_to_label[label_id]
            
            if label.startswith('B-'):
                # Save previous marker if exists
                if current_marker:
                    markers.append(current_marker)
                
                # Start new marker
                marker_type = label[2:]  # Remove 'B-'
                current_marker = {
                    'startIndex': int(offset[0]),
                    'endIndex': int(offset[1]),
                    'type': marker_type
                }
            
            elif label.startswith('I-'):
                # Continue current marker
                if current_marker and label[2:] == current_marker['type']:
                    current_marker['endIndex'] = int(offset[1])
            
            else:  # 'O' label
                # Save previous marker if exists
                if current_marker:
                    markers.append(current_marker)
                    current_marker = None
        
        # Save last marker if exists
        if current_marker:
            markers.append(current_marker)
        
        all_predictions.append(markers)
    
    return all_predictions

print('Converting predictions to character spans...')
test_offset_mappings = [test_ds_tokenized[i]['offset_mapping'] for i in range(len(test_ds_tokenized))]
test_texts = [test_data_original[i]['text'] for i in range(len(test_data_original))]

predicted_markers = extract_markers_from_predictions(
    test_pred_labels, 
    test_offset_mappings, 
    test_texts,
    id_to_label
)

total_markers = sum(len(m) for m in predicted_markers)
print(f'✓ Extracted {total_markers} markers across {len(predicted_markers)} examples')

Converting predictions to character spans...
✓ Extracted 259 markers across 100 examples


In [78]:
# Create submission file and ZIP
import zipfile

MARKERS_DIR = Path('..')  # Markers-Extraction directory
SUBMISSION_FILE = MARKERS_DIR / 'submission.jsonl'
SUBMISSION_ZIP = MARKERS_DIR / 'submission.zip'

print(f'Creating submission file: {SUBMISSION_FILE}')
with open(SUBMISSION_FILE, 'w') as f:
    for i, example in enumerate(test_data_original):
        submission_entry = {
            '_id': example['_id'],
            'markers': predicted_markers[i]
        }
        f.write(json.dumps(submission_entry) + '\n')

print(f'✅ Submission file created: {SUBMISSION_FILE}')
print(f'   Total examples: {len(test_data_original)}')
print(f'   Total markers: {total_markers}')

# Create ZIP file
print(f'\nCreating ZIP file: {SUBMISSION_ZIP}')
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(SUBMISSION_FILE, arcname='submission.jsonl')

print(f'✅ ZIP file created: {SUBMISSION_ZIP}')
print(f'\n{"="*70}')
print(f'🎉 SUBMISSION READY!')
print(f'{"="*70}')
print(f'📦 File to upload: {SUBMISSION_ZIP}')
print(f'🌐 Upload to: https://www.codabench.org/competitions/10751/')
print(f'{"="*70}')

Creating submission file: ../submission.jsonl
✅ Submission file created: ../submission.jsonl
   Total examples: 100
   Total markers: 259

Creating ZIP file: ../submission.zip
✅ ZIP file created: ../submission.zip

🎉 SUBMISSION READY!
📦 File to upload: ../submission.zip
🌐 Upload to: https://www.codabench.org/competitions/10751/


In [79]:
# Show sample predictions
print('='*80)
print('SAMPLE PREDICTIONS')
print('='*80)

for i in range(min(3, len(test_data_original))):
    example = test_data_original[i]
    markers = predicted_markers[i]
    
    print(f'\n📄 Example {i+1} (ID: {example["_id"]})')
    print(f'Text: {example["text"][:100]}...')
    print(f'Predicted {len(markers)} markers:')
    
    for marker in markers[:5]:  # Show first 5 markers
        text_snippet = example["text"][marker["startIndex"]:marker["endIndex"]]
        print(f'  • {marker["type"]}: "{text_snippet}" ({marker["startIndex"]}-{marker["endIndex"]})')
    
    if len(markers) > 5:
        print(f'  ... and {len(markers) - 5} more markers')
    print('-'*80)

SAMPLE PREDICTIONS

📄 Example 1 (ID: t1_emz6exn)
Text: So according to Trump deeps state John Kerry is directing Iran on what to do. The Obama admin got Ir...
Predicted 7 markers:
  • Actor: "John Kerry" (34-44)
  • Action: "directing Iran on what to do." (48-77)
  • Actor: "Obama admin" (82-93)
  • Action: "got Iran all set up the way they" (94-126)
  • Action: "air dropping billions in bribe money to Iran to pay them to set up their deep state agenda" (140-230)
  ... and 2 more markers
--------------------------------------------------------------------------------

📄 Example 2 (ID: t1_f07ejkp)
Text: RFK Jr. claims that CIA operatives destroyed evidence in his father’s assassination that clearly imp...
Predicted 5 markers:
  • Actor: "RFK Jr" (0-6)
  • Actor: "CIA operatives" (20-34)
  • Action: "destroyed evidence" (35-53)
  • Victim: "Eugene Thane" (108-120)
  • Evidence: "social media post was published" (128-159)
-------------------------------------------------------------------